# XLM-RoBERTa · 한·영 17종 다중 라벨

원본 `finetuned_bert (2).ipynb`의 0-based cell 15를 변경 없이 추출했습니다.

원본 SHA-256: `e052814403adca3379fa0fd8c4731970940a415488595e3c1b5db277cc29006c`

원본 실행 출력·위젯 메타데이터·원시 데이터는 포함하지 않습니다. 저장된 과거 결과와 입력 준비 방법은 README를 참고하세요. 이 공개 사본은 재학습하지 않았으며, 실행에는 별도 CSV가 필요합니다. v4 라벨 매핑과 분할 중복 검토 후 재평가가 필요합니다.


In [ ]:
# ==========================================
# 1. 라이브러리 임포트
# ==========================================
# !pip install transformers datasets accelerate -U  # (필요시 주석 해제)

import torch
import numpy as np
import re
from dataclasses import dataclass
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback  # 👈 이 부분이 반드시 들어가야 합니다!
)
from sklearn.metrics import label_ranking_average_precision_score, f1_score
from datasets import load_dataset

# ==========================================
# 2. 설정 (XLM-RoBERTa로 변경 ⭐)
# ==========================================
# BERT 대신 성능이 더 좋다고 알려진 XLM-RoBERTa 사용
model_name = 'xlm-roberta-base'

# 17개 라벨 (순서 그대로 유지)
labels = [
    '여성/가족', '남성', '성소수자', '인종/국적', '연령',
    '지역', '종교', '기타 혐오', '악플/욕설', 'clean',
    'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'clean_en'
]
num_labels = len(labels)

# ==========================================
# 3. 데이터셋 로드 및 토크나이징
# ==========================================
data_files = {
    "train": "integrated_train_v4.csv",
    "valid": "integrated_valid_v4.csv"
}
dataset = load_dataset("csv", data_files=data_files)

# XLM-RoBERTa용 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

def parse_labels_str(s):
    if not isinstance(s, str): return s
    s = s.strip().replace("[", "").replace("]", "")
    parts = re.split(r"[,\s]+", s)
    return [int(p) for p in parts if p != ""]

def preprocess(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=128,
    )
    raw_labels = examples["labels"]
    processed_labels = [parse_labels_str(s) for s in raw_labels]
    tokenized["labels"] = processed_labels
    return tokenized

print("XLM-RoBERTa 토크나이저로 변환 중...")
tokenized_dataset = dataset.map(preprocess, batched=True)

# ==========================================
# 4. Custom Collator
# ==========================================
@dataclass
class DataCollatorWithFloatLabels(DataCollatorWithPadding):
    def __call__(self, features):
        labels = [f["labels"] for f in features]
        for f in features:
            f.pop("labels")
        batch = super().__call__(features)
        label_tensors = [torch.tensor(l, dtype=torch.float) for l in labels]
        batch["labels"] = torch.stack(label_tensors)
        return batch

collator = DataCollatorWithFloatLabels(tokenizer=tokenizer)

# ==========================================
# 5. 평가 지표
# ==========================================
def compute_metrics(x):
    logits = x.predictions
    labels = x.label_ids
    probs = 1 / (1 + np.exp(-logits))

    lrap = label_ranking_average_precision_score(labels, probs)
    y_pred = (probs >= 0.5).astype(int)
    micro_f1 = f1_score(labels, y_pred, average="micro", zero_division=0)

    return {"lrap": lrap, "micro_f1": micro_f1}
# ==========================================
# 6. 모델 준비 및 학습 시작 (모니터링 & 조기종료 추가!)
# ==========================================
print(f"모델 로드 중: {model_name}...")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

model.config.id2label = {i: l for i, l in enumerate(labels)}
model.config.label2id = {l: i for i, l in enumerate(labels)}

train_args = TrainingArguments(
    output_dir="./xlm_roberta_hate_model_v4",
    eval_strategy="epoch",        # 매 에포크마다 평가
    save_strategy="epoch",        # 매 에포크마다 모델 저장
    logging_strategy="steps",     # 👈 스텝마다 로그 기록 (그래프를 촘촘하게 그리기 위함)
    logging_steps=100,            # 👈 100스텝마다 점수 기록
    logging_dir='./logs',         # 👈 텐서보드 그래프가 저장될 폴더
    report_to="tensorboard",      # 👈 텐서보드 사용 명시
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,           # 👈 [변경] 에포크를 3에서 5(또는 그 이상)로 넉넉히 늘립니다!
    load_best_model_at_end=True,  # 👈 학습 종료 시 가장 점수가 높았던 최고의 모델을 불러옴
    metric_for_best_model="lrap", # 👈 최고 모델을 가리는 기준 (LRAP 점수)
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    compute_metrics=compute_metrics,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # 👈 [추가됨] 2번의 에포크 동안 LRAP 점수가 안 오르면 자동 스탑!
)

print("학습 시작! (XLM-RoBERTa + TensorBoard) 🚀")
trainer.train()

# ==========================================
# 7. 모델 저장
# ==========================================
save_path = "./final_xlm_roberta_model_v4" # 👈 여기도 _v2 추가
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"모델 저장 완료: {save_path}")